# 07 — Evaluate the three horizon-specific adapters

This notebook requires one current Qwen3-4B adapter for each horizon. It verifies panel and baseline checksums, evaluates at least 1,000 matched untouched-test cases per horizon, reports accuracy by table and forecast scope, and compares each adapter only with its matching 1Q, 2Q, or 4Q baselines.


## 1. Mount the Colab project

Connect Google Drive and point the notebook at the same persistent project used by notebook 06.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/JobAI")
os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"

## 2. Install the evaluation stack

Install the same pinned model libraries used for fine-tuning. The base model itself is not downloaded again.

In [ ]:
%pip install --quiet -r requirements-train-colab.txt

## 3. Discover the three comparable adapter runs

The notebook selects the latest current-dataset `enhanced_v1` run for `(1,)`, `(2,)`, and `(4,)`. It refuses stale adapters whose recorded panel checksum differs from the current Notebook 05 dataset card.


In [ ]:
import hashlib, importlib.metadata, json, os, platform, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt

def find_repo():
    configured = os.environ.get("JOBAI_REPO")
    if configured:
        return Path(configured).resolve()
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "model.yaml").is_file():
            return candidate
    return current

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

REPO = find_repo()
PRO = REPO / "data" / "processed"
MAN = REPO / "data" / "manifests"
REPORTS = REPO / "reports"
FIGURES = REPORTS / "figures"
EVALUATION_ROOT = REPORTS / "model_evaluations"
FIGURES.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CFG = yaml.safe_load((REPO / "configs" / "model.yaml").read_text())
SEED = int(MODEL_CFG["training"]["seed"])
TEST_PATH = PRO / "panel_test.jsonl"
BASELINE_PATH = REPORTS / "baseline_predictions.csv"
BASELINE_MANIFEST_PATH = REPORTS / "baseline_run_manifest.json"
CARD_PATH = MAN / "panel_dataset_card.json"
RUN_MANIFEST_DIR = MAN / "finetune_runs"
for required in (TEST_PATH, BASELINE_PATH, BASELINE_MANIFEST_PATH, CARD_PATH):
    assert required.is_file(), f"Missing prerequisite: {required}"
assert RUN_MANIFEST_DIR.is_dir(), "Run notebook 06 for h1, h2, and h4 first"
panel_card = json.loads(CARD_PATH.read_text())
baseline_manifest = json.loads(BASELINE_MANIFEST_PATH.read_text())
assert panel_card["files"]["test"]["sha256"] == sha256(TEST_PATH), "Notebook 05 test checksum is stale"
assert baseline_manifest["outputs"]["reports/baseline_predictions.csv"]["sha256"] == sha256(BASELINE_PATH), "Notebook 04 predictions are stale"
assert baseline_manifest["selection_catalog_sha256"] == panel_card["selection_catalog_sha256"], "Notebooks 04 and 05 used different target catalogs"
current_card_sha = sha256(CARD_PATH)

available_runs = []
for manifest_path in sorted(RUN_MANIFEST_DIR.glob("*.json")):
    manifest = json.loads(manifest_path.read_text())
    adapter_dir = REPO / manifest.get("adapter_path", "")
    if manifest.get("base_model") != "Qwen/Qwen3-4B" or not adapter_dir.is_dir():
        continue
    if manifest.get("panel_dataset_card_sha256") != current_card_sha:
        continue
    training_cfg = manifest.get("model_config", {}).get("training", {})
    feature_set = manifest.get("feature_set", training_cfg.get("feature_set", "legacy_v1"))
    horizons = tuple(sorted(int(h) for h in manifest.get("training_horizons", training_cfg.get("training_horizons", []))))
    available_runs.append({**manifest, "manifest_path": manifest_path, "adapter_dir": adapter_dir,
                           "feature_set": feature_set, "training_horizons": horizons,
                           "train_count": int(manifest.get("train_examples_used", manifest.get("train_examples", -1))),
                           "max_seq_length": int(training_cfg.get("max_seq_length", 512))})

selected_runs = []
for horizon in (1, 2, 4):
    matches = [run for run in available_runs if run["feature_set"] == "enhanced_v1" and run["training_horizons"] == (horizon,)]
    assert matches, f"Missing current-dataset enhanced Qwen3-4B h{horizon} adapter. Run Notebook 06 with that profile."
    selected_runs.append(sorted(matches, key=lambda run: run["run_id"])[-1])

MODEL_ID = "Qwen/Qwen3-4B"
candidate_by_id = {candidate["id"]: candidate for candidate in MODEL_CFG["candidates"]}
MODEL_CHOICE = candidate_by_id[MODEL_ID]
MODEL_ARCHITECTURE = MODEL_CHOICE.get("architecture", "causal_lm")
BASE_MODEL_DIR = REPO / "models" / "base" / MODEL_ID.replace("/", "--")
assert (BASE_MODEL_DIR / ".download_complete").is_file(), f"Persistent base model missing: {BASE_MODEL_DIR}"
assert torch.cuda.is_available(), "A CUDA GPU is required for model evaluation"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = bool(torch.cuda.is_bf16_supported())
for run in selected_runs:
    horizon = run["training_horizons"][0]
    run["model_label"] = f"qwen3_4b_h{horizon}_{run['train_count']}_balanced_examples"
    print(f"selected h{horizon}: {run['run_id']} | examples={run['train_count']:,} | max_seq={run['max_seq_length']}")
print("GPU:", GPU_NAME, f"({GPU_VRAM_GB:.1f} GB)")


## 4. Build a large matched untouched-test sample

Each horizon receives up to 1,000 deterministic test cases for which all four baselines exist. This is large enough for a more stable decision than the previous 200-case comparison and keeps all model/baseline cases identical.


In [ ]:
TEST_PER_HORIZON = 1000
KEYS = ["table_id", "series_id", "horizon_q", "origin_quarter", "target_quarter"]
test_full = pd.read_json(TEST_PATH, lines=True)
assert (test_full["split"] == "test").all()
baseline_full = pd.read_csv(BASELINE_PATH)
baseline_test = baseline_full.loc[baseline_full["split"] == "test"].copy()
required_models = {"last_value", "seasonal_naive", "ridge", "ridge_enhanced"}
found_models = set(baseline_test["model"].unique())
assert required_models.issubset(found_models), f"Rerun Notebook 04; missing baselines: {sorted(required_models - found_models)}"
model_sets = baseline_test.groupby(KEYS)["model"].agg(set)
eligible_keys = model_sets[model_sets.map(lambda values: required_models.issubset(values))].reset_index()[KEYS]
eligible = test_full.merge(eligible_keys, on=KEYS, how="inner", validate="one_to_one")

def deterministic_group_sample(frame, n, seed):
    ranked = frame.copy()
    ranked["_sample_key"] = ranked["example_id"].map(lambda value: hashlib.sha256(f"{seed}|{value}".encode()).hexdigest())
    # Round-robin by table and scope prevents one large target family from dominating evaluation.
    ranked["_stratum"] = ranked[["table_id", "forecast_scope"]].astype(str).agg("|".join, axis=1)
    ranked["_round"] = ranked.groupby("_stratum")["_sample_key"].rank(method="first").astype(int)
    ranked["_stratum_key"] = ranked["_stratum"].map(lambda value: hashlib.sha256(f"{seed}|{value}".encode()).hexdigest())
    return ranked.sort_values(["_round", "_stratum_key", "_sample_key"]).head(min(n, len(ranked))).drop(
        columns=["_sample_key", "_stratum", "_round", "_stratum_key"]
    )

sample_parts = [deterministic_group_sample(group, TEST_PER_HORIZON, SEED + int(horizon))
                for horizon, group in eligible.groupby("horizon_q", sort=True)]
test_sample = pd.concat(sample_parts, ignore_index=True).sort_values(["horizon_q", "table_id", "series_id", "origin_quarter"]).reset_index(drop=True)
assert set(test_sample["horizon_q"]) == {1, 2, 4}
assert not test_sample.duplicated("example_id").any()
print("full untouched test examples:", len(test_full))
print("eligible matched examples   :", len(eligible))
display(test_sample.groupby(["horizon_q", "table_id", "forecast_scope"]).size().rename("selected").reset_index())


## 5. Load the base model once and attach all three adapters

Load the shared Qwen3-4B base model in four-bit mode only once. Attach the 1Q, 2Q, and 4Q LoRA adapters under distinct names so evaluation can switch horizons without loading another copy of the 4B base weights.


In [ ]:
from transformers import AutoModelForCausalLM, AutoModelForMultimodalLM, AutoProcessor, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

compute_dtype = torch.bfloat16 if BF16 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
)
first_adapter = selected_runs[0]["adapter_dir"]
if MODEL_ARCHITECTURE == "multimodal_text_only":
    processor = AutoProcessor.from_pretrained(first_adapter, local_files_only=True)
    tokenizer = processor.tokenizer
    model_class = AutoModelForMultimodalLM
else:
    processor = None
    tokenizer = AutoTokenizer.from_pretrained(first_adapter, local_files_only=True, use_fast=True)
    model_class = AutoModelForCausalLM
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = model_class.from_pretrained(
    BASE_MODEL_DIR, local_files_only=True, quantization_config=quantization_config,
    device_map="auto", dtype=compute_dtype,
)
first_name = selected_runs[0]["model_label"]
model = PeftModel.from_pretrained(base_model, first_adapter, adapter_name=first_name, local_files_only=True)
for run in selected_runs[1:]:
    model.load_adapter(run["adapter_dir"], adapter_name=run["model_label"], local_files_only=True)
model.eval()
print("Loaded one base model with adapters:", [run["model_label"] for run in selected_runs])


## 6. Generate matched forecasts from all three adapters

Each adapter sees only rows for its own horizon. Prompts exactly reproduce Notebook 06, including source family, scope, measure, engineered features, and compact origin-safe hierarchy context. Parse failures remain visible and are never replaced with baseline predictions.


In [ ]:
SYSTEM_TEXT = ("You forecast Finnish job-vacancy time series from Statistics Finland and KEHA. "
               "Return exactly one JSON object with one numeric field named target_scaled_change.")
GENERATION_BATCH_SIZE = 16

def format_number(value):
    return format(float(value), ".6g")

FEATURE_ALIASES = [
    ("quarter_of_year", "quarter"), ("qoq_change_scaled", "qoq"),
    ("yoy_change_scaled", "yoy"), ("recent_mean_4_scaled", "mean4"),
    ("window_mean_scaled", "mean_window"), ("recent_slope_4_scaled", "slope4"),
    ("window_slope_scaled", "slope_window"), ("recent_std_4_scaled", "std4"),
    ("window_std_scaled", "std_window"), ("zero_fraction_window", "zero_fraction"),
]

def compact_feature_text(engineered):
    return "; ".join(f"{alias}={format_number(engineered[key])}" for key, alias in FEATURE_ALIASES)

def prompt_messages(row, run):
    history = json.loads(row["input_values_json"])
    dimensions = json.loads(row["dimensions_json"])
    engineered = json.loads(row["engineered_features_json"])
    context_raw = row.get("hierarchical_context_json", "{}")
    context = json.loads(context_raw) if isinstance(context_raw, str) and context_raw else {}
    lines = [
        f"Source family: {row['series_family']}",
        f"Table: {row['table_id']}",
        f"Forecast scope: {row['forecast_scope']}",
        f"Measure: {row['measure_code']}",
        f"Dimensions: {json.dumps(dimensions, ensure_ascii=False, sort_keys=True)}",
        f"History start: {row['window_start_quarter']}",
        f"{len(history)} quarterly values, oldest to newest: {[format_number(v) for v in history]}",
        f"Origin-safe features: {compact_feature_text(engineered)}",
    ]
    if context:
        lines.append(f"Reference context: {json.dumps(context, ensure_ascii=False, sort_keys=True)}")
    lines.extend([
        f"Forecast origin: {row['origin_quarter']}",
        f"Forecast horizon: {int(row['horizon_q'])} quarter(s)",
        f"Target quarter: {row['target_quarter']}",
        f"Scale: {format_number(row['scale'])}",
        "Predict target_scaled_change = (target vacancy value - latest history value) / scale.",
    ])
    return [{"role": "system", "content": SYSTEM_TEXT}, {"role": "user", "content": chr(10).join(lines)}]

def parse_scaled_change(text):
    match = re.search(r"\{[^{}]*\}", text)
    if not match:
        return None
    try:
        value = float(json.loads(match.group(0))["target_scaled_change"])
        return value if np.isfinite(value) else None
    except (KeyError, TypeError, ValueError, json.JSONDecodeError):
        return None

prediction_rows = []
generation_seconds_by_run = {}
all_records = test_sample.to_dict(orient="records")
for run in selected_runs:
    records = [row for row in all_records if int(row["horizon_q"]) in run["training_horizons"]]
    model_label = run["model_label"]
    model.set_adapter(model_label)
    run_max_length = int(run["max_seq_length"])
    prompt_probe = [tokenizer.apply_chat_template(prompt_messages(row, run), tokenize=False,
                    add_generation_prompt=True, enable_thinking=False) for row in records]
    prompt_lengths = [len(tokenizer(prompt, add_special_tokens=False)["input_ids"]) for prompt in prompt_probe]
    print(f"{model_label}: prompt tokens median/max={int(np.median(prompt_lengths))}/{max(prompt_lengths)}, limit={run_max_length}")
    assert max(prompt_lengths) <= run_max_length
    started_at = time.time()
    for start in range(0, len(records), GENERATION_BATCH_SIZE):
        batch = records[start:start + GENERATION_BATCH_SIZE]
        prompts = [tokenizer.apply_chat_template(prompt_messages(row, run), tokenize=False, add_generation_prompt=True, enable_thinking=False) for row in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=run_max_length).to(model.device)
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=32, do_sample=False, use_cache=True)
        prompt_width = inputs["input_ids"].shape[1]
        responses = tokenizer.batch_decode(generated[:, prompt_width:], skip_special_tokens=True)
        for row, response in zip(batch, responses):
            predicted_change = parse_scaled_change(response.strip())
            parsed = predicted_change is not None
            predicted_value = max(0.0, float(row["last_value"]) + predicted_change * float(row["scale"])) if parsed else np.nan
            prediction_rows.append({**{key: row[key] for key in ["example_id", *KEYS, "series_family", "forecast_scope", "measure_code"]},
                                    "run_id": run["run_id"], "train_examples": run["train_count"], "feature_set": run["feature_set"],
                                    "training_horizons": "-".join(map(str, run["training_horizons"])),
                                    "model": model_label, "y_true": float(row["target_value"]),
                                    "y_pred": predicted_value, "parsed": parsed, "response": response.strip()})
        print(f"{model_label}: generated {min(start + len(batch), len(records)):,}/{len(records):,}", end=chr(13))
    generation_seconds_by_run[run["run_id"]] = time.time() - started_at
    run_predictions = [row for row in prediction_rows if row["run_id"] == run["run_id"]]
    print()
    print(f"{model_label}: {generation_seconds_by_run[run['run_id']] / 60:.1f} minutes, parse success={np.mean([row['parsed'] for row in run_predictions]):.1%}")
model_predictions = pd.DataFrame(prediction_rows)


## 7. Calculate matched metrics by horizon and product scope

The overall table is useful, but it can hide a weak national model behind thousands of detailed KEHA cases. This section therefore reports the same MAE, RMSE, MASE, and sMAPE separately for each table and forecast scope.


In [ ]:
sample_keys = test_sample[KEYS].drop_duplicates()
matched_baselines = baseline_test.merge(sample_keys, on=KEYS, how="inner", validate="many_to_one")
scale_lookup = matched_baselines.drop_duplicates(KEYS)[KEYS + ["mase_scale"]]
model_scored = model_predictions.merge(scale_lookup, on=KEYS, how="left", validate="many_to_one")
assert model_scored["mase_scale"].notna().all()
model_scored["split"] = "test"
model_scored["error"] = model_scored["y_true"] - model_scored["y_pred"]
model_scored["abs_error"] = model_scored["error"].abs()
model_scored["squared_error"] = model_scored["error"].pow(2)
model_scored["scaled_abs_error"] = model_scored["abs_error"] / model_scored["mase_scale"]
denom = (model_scored["y_true"].abs() + model_scored["y_pred"].abs()) / 2
model_scored["smape_component_pct"] = np.where(denom == 0, 0.0, 100 * model_scored["abs_error"] / denom)
metric_columns = ["model", "table_id", "series_family", "forecast_scope", "measure_code", "series_id", "horizon_q", "y_true", "y_pred", "abs_error", "squared_error", "scaled_abs_error", "smape_component_pct"]
combined = pd.concat([matched_baselines[metric_columns], model_scored.loc[model_scored["parsed"], metric_columns]], ignore_index=True)

def metric_summary(group):
    return pd.Series({"n": len(group), "MAE": group["abs_error"].mean(),
                      "RMSE": np.sqrt(group["squared_error"].mean()),
                      "MASE": group["scaled_abs_error"].mean(),
                      "sMAPE_pct": group["smape_component_pct"].mean()})

by_series = (combined.groupby(["model", "table_id", "series_family", "forecast_scope", "measure_code", "series_id", "horizon_q"], sort=True)
             .apply(metric_summary, include_groups=False).reset_index())
summary = (by_series.groupby(["model", "horizon_q"], sort=True)
           [["MAE", "RMSE", "MASE", "sMAPE_pct"]].mean().reset_index())
summary["n_series"] = by_series.groupby(["model", "horizon_q"])["series_id"].nunique().to_numpy()
parse_rates = model_predictions.groupby("model")["parsed"].mean().to_dict()
training_counts = model_predictions.groupby("model")["train_examples"].first().to_dict()
summary["parse_success"] = summary["model"].map(parse_rates).fillna(1.0)
summary["train_examples"] = summary["model"].map(training_counts)
display(summary.round(3))

by_scope = (by_series.groupby(["model", "table_id", "forecast_scope", "horizon_q"], sort=True)
            .agg(MAE=("MAE", "mean"), RMSE=("RMSE", "mean"), MASE=("MASE", "mean"),
                 sMAPE_pct=("sMAPE_pct", "mean"), n_series=("series_id", "nunique")).reset_index())
display(by_scope.round(3))


## 8. Compare adapters, baselines, and save separate evidence

For every horizon, compare its dedicated adapter with the strongest matching baseline. Save aggregate results plus a separate evidence folder and evaluation manifest for each immutable training run, so one evaluation never overwrites another.


In [ ]:
adapter_labels = [run["model_label"] for run in selected_runs]
best_baseline = (summary.loc[~summary["model"].isin(adapter_labels)]
                 .sort_values(["horizon_q", "MAE"]).groupby("horizon_q", as_index=False).first())
comparison_parts = []
for run in selected_runs:
    label = run["model_label"]
    adapter_summary = summary.loc[summary["model"] == label, ["horizon_q", "MAE", "RMSE", "MASE", "sMAPE_pct", "parse_success", "train_examples"]]
    part = adapter_summary.merge(best_baseline[["horizon_q", "model", "MAE"]], on="horizon_q", suffixes=("_adapter", "_best_baseline"))
    part["run_id"] = run["run_id"]
    part["adapter_model"] = label
    part["MAE_improvement_pct"] = 100 * (part["MAE_best_baseline"] - part["MAE_adapter"]) / part["MAE_best_baseline"]
    part["adapter_beats_best_baseline"] = part["MAE_adapter"] < part["MAE_best_baseline"]
    comparison_parts.append(part)
adapter_vs_baselines = pd.concat(comparison_parts, ignore_index=True)
adapter_only = summary.loc[summary["model"].isin(adapter_labels)].copy()
adapter_only["best_adapter_MAE"] = adapter_only.groupby("horizon_q")["MAE"].transform("min") == adapter_only["MAE"]
display(adapter_only.sort_values(["horizon_q", "MAE"]).round(3))
display(adapter_vs_baselines.round(3))

# Diagnose whether errors come from table family, target scale, zeros, or a few difficult series.
diagnostic_rows = pd.concat([
    matched_baselines.assign(parsed=True, run_id="baseline", train_examples=np.nan),
    model_scored,
], ignore_index=True, sort=False)
diagnostic_rows["target_size_band"] = pd.cut(
    diagnostic_rows["y_true"], bins=[-np.inf, 0, 25, 100, 500, np.inf],
    labels=["zero", "1-25", "26-100", "101-500", "over-500"]
)
diagnostic_rows["is_zero_target"] = diagnostic_rows["y_true"].eq(0)
error_diagnostics = (diagnostic_rows.groupby(
    ["model", "horizon_q", "table_id", "target_size_band"], observed=True, dropna=False
).agg(n=("y_true", "size"), n_series=("series_id", "nunique"),
      target_median=("y_true", "median"), median_AE=("abs_error", "median"),
      MAE=("abs_error", "mean"), RMSE=("squared_error", lambda values: float(np.sqrt(values.mean()))),
      MASE=("scaled_abs_error", "mean"), sMAPE_pct=("smape_component_pct", "mean"),
      parse_success=("parsed", "mean")).reset_index())
worst_series = (diagnostic_rows.groupby(["model", "horizon_q", "table_id", "series_id"], dropna=False)
                .agg(n=("y_true", "size"), target_median=("y_true", "median"),
                     median_AE=("abs_error", "median"), MAE=("abs_error", "mean"),
                     max_AE=("abs_error", "max")).reset_index()
                .sort_values(["model", "horizon_q", "MAE"], ascending=[True, True, False]))
display(error_diagnostics.round(3))

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

comparison_spec = {
    "selected_run_ids": [run["run_id"] for run in selected_runs],
    "selected_feature_sets": [run["feature_set"] for run in selected_runs],
    "selected_max_seq_lengths": [run["max_seq_length"] for run in selected_runs],
    "panel_test_sha256": sha256(TEST_PATH),
    "baseline_predictions_sha256": sha256(BASELINE_PATH),
    "test_examples_per_horizon": TEST_PER_HORIZON,
}
comparison_id = hashlib.sha256(json.dumps(comparison_spec, sort_keys=True).encode()).hexdigest()[:12]
AGGREGATE_DIR = EVALUATION_ROOT / "comparisons" / f"comparison_{comparison_id}"
AGGREGATE_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATE_PREDICTIONS_PATH = AGGREGATE_DIR / "adapter_test_predictions.csv"
AGGREGATE_SUMMARY_PATH = AGGREGATE_DIR / "adapter_and_baseline_metrics.csv"
ADAPTER_COMPARISON_PATH = AGGREGATE_DIR / "adapter_training_size_comparison.csv"
BASELINE_COMPARISON_PATH = AGGREGATE_DIR / "adapter_vs_baselines.csv"
ERROR_DIAGNOSTICS_PATH = AGGREGATE_DIR / "error_diagnostics.csv"
WORST_SERIES_PATH = AGGREGATE_DIR / "worst_series.csv"
SCOPE_METRICS_PATH = AGGREGATE_DIR / "metrics_by_scope.csv"
model_scored.to_csv(AGGREGATE_PREDICTIONS_PATH, index=False)
summary.to_csv(AGGREGATE_SUMMARY_PATH, index=False)
adapter_only.to_csv(ADAPTER_COMPARISON_PATH, index=False)
adapter_vs_baselines.to_csv(BASELINE_COMPARISON_PATH, index=False)
error_diagnostics.to_csv(ERROR_DIAGNOSTICS_PATH, index=False)
worst_series.to_csv(WORST_SERIES_PATH, index=False)
by_scope.to_csv(SCOPE_METRICS_PATH, index=False)

comparison_manifest_path = AGGREGATE_DIR / "comparison_manifest.json"
comparison_manifest = {**comparison_spec, "comparison_id": comparison_id,
                       "models": sorted(summary["model"].unique()),
                       "output_directory": str(AGGREGATE_DIR.relative_to(REPO))}
comparison_manifest_path.write_text(json.dumps(comparison_manifest, indent=2, ensure_ascii=False))
LATEST_COMPARISON_PATH = EVALUATION_ROOT / "latest_comparison_manifest.json"
LATEST_COMPARISON_PATH.write_text(json.dumps(comparison_manifest, indent=2, ensure_ascii=False))

for run in selected_runs:
    label = run["model_label"]
    run_dir = EVALUATION_ROOT / run["run_id"]
    run_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = run_dir / "test_predictions.csv"
    series_path = run_dir / "metrics_by_series.csv"
    summary_path = run_dir / "metrics_vs_baselines.csv"
    comparison_path = run_dir / "decision_vs_best_baseline.csv"
    run_prediction_rows = model_scored.loc[model_scored["run_id"] == run["run_id"]]
    run_series = by_series.loc[by_series["model"] == label]
    run_summary = pd.concat([summary.loc[summary["model"] == label], summary.loc[~summary["model"].isin(adapter_labels)]], ignore_index=True)
    run_comparison = adapter_vs_baselines.loc[adapter_vs_baselines["run_id"] == run["run_id"]]
    run_prediction_rows.to_csv(prediction_path, index=False)
    run_series.to_csv(series_path, index=False)
    run_summary.to_csv(summary_path, index=False)
    run_comparison.to_csv(comparison_path, index=False)
    parse_rate = float(run_prediction_rows["parsed"].mean())
    passed = bool(parse_rate >= 0.99 and run_comparison["adapter_beats_best_baseline"].all())
    evaluation_manifest = {
        "status": "pilot_passed" if passed else "pilot_failed",
        "run_id": run["run_id"], "training_manifest": str(run["manifest_path"].relative_to(REPO)),
        "base_model": MODEL_ID, "adapter_path": str(run["adapter_dir"].relative_to(REPO)),
        "train_examples": run["train_count"], "feature_set": run["feature_set"],
        "training_horizons": list(run["training_horizons"]),
        "test_examples_available": len(test_full), "test_examples_used": len(test_sample),
        "test_examples_per_horizon": TEST_PER_HORIZON, "test_data_used_for_training": False,
        "parse_success_rate": parse_rate, "minimum_parse_success_rate": 0.99,
        "adapter_beats_best_baseline_all_horizons": bool(run_comparison["adapter_beats_best_baseline"].all()),
        "generation_seconds": generation_seconds_by_run[run["run_id"]], "gpu": GPU_NAME,
        "outputs": {str(path.relative_to(REPO)): {"sha256": sha256(path)} for path in [prediction_path, series_path, summary_path, comparison_path]},
    }
    evaluation_manifest_path = run_dir / "evaluation_manifest.json"
    evaluation_manifest_path.write_text(json.dumps(evaluation_manifest, indent=2, ensure_ascii=False))
    print("saved run evidence:", run_dir)
print("saved aggregate comparison:", AGGREGATE_DIR)


## 9. Compare test MAE visually

Each colored Qwen bar now represents a different horizon-specific adapter. Shorter bars are better. Use the by-scope report—not this aggregate chart alone—to decide whether national, regional, occupation, industry, and detailed-geography forecasts are all ready.


In [ ]:
%matplotlib inline
plot_data = summary.pivot(index="horizon_q", columns="model", values="MAE").sort_index()
ax = plot_data.plot(kind="bar", figsize=(13, 6), width=0.82)
ax.set(title="Untouched-test comparison: Qwen3-4B horizon-specific adapters and baselines", xlabel="Forecast horizon (quarters)", ylabel="Macro-average MAE")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
FIGURE_PATH = FIGURES / "qwen3_4b_horizon_adapters_vs_baselines_mae.png"
plt.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()
print("saved:", FIGURE_PATH)


## Decision rule

Deployment requires all three adapters, at least 99% parse success, and improvement over the strongest matching baseline for each horizon. Review `metrics_by_scope.csv` separately: a good overall average does not compensate for an unusable national or regional forecast family. RAG is evaluated later and may explain a forecast, but it cannot repair a failed numeric adapter.
